<a href="https://colab.research.google.com/github/Diwash17/FlyRank-AI-Assentment-and-Capstone/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mirzaAsca/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Clustering.** The decision this improves: which pages should get grouped together so an
editor can spot content coverage gaps and cannibalization at a glance, instead of reading
30,000 rows one at a time. There's no outcome to predict here (no "will this decline" label
I'm targeting) — I want to find the natural groupings that already exist in how pages look
and perform, then hand editors a small number of groups instead of a spreadsheet.

Who acts on it: a content strategist deciding where to consolidate near-duplicate pages
(cannibalization) or where a topic has thin/no coverage (a gap). A wrong grouping costs editor
time chasing a "gap" that isn't real, or missing a real overlap — so I'll only claim
"directional, decision-support" groupings, not hard categories.


In [4]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/Diwash17/FlyRank-AI-Assentment-and-Capstone/main/data/raw/content_refresh_anonymized.csv")
df[["content_type", "main_intent"]].value_counts()

content_type        main_intent  
keyword article     informational    16538
                    transactional     5733
                    commercial        4612
comparison article  informational      697
keyword article     navigational        46
Name: count, dtype: int64

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**No target — this is unsupervised.** Clustering doesn't predict a label; it discovers
structure. The closest thing to a "proxy" is the **cluster assignment itself**, which becomes
a new categorical column I hand back to editors (e.g. `cluster_0`..`cluster_3`).

I deliberately leave `trend_direction` and `trend_pct` out of the features that produce the
clusters — the data dictionary is explicit that these are the *label* for the classification
lane, never a feature. Even though clustering has no label to leak into, keeping them out
keeps this a genuinely separate lane: I'm grouping by what a page **is and does**
(content shape + performance), not by whether it's currently rising or falling.


In [5]:
# No label column to inspect -- confirming there's nothing I'm quietly treating as a target.
# trend_direction / trend_pct exist in the data but are reserved for the classification lane.
print("trend_direction" in df.columns, "trend_pct" in df.columns)
print("-> present in the data, but excluded from my feature list below on purpose.")


True True
-> present in the data, but excluded from my feature list below on purpose.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Silhouette score, plus a human sense-check.** Silhouette score (I'll compute mine below)
tells me whether the clusters are numerically well-separated, on a -1..1 scale where higher
means tighter, more distinct groups. But silhouette alone can't tell an editor a cluster is
*useful* — a statistically clean cluster of "long informational articles with near-zero
traffic" is only worth acting on if a human looking at real examples agrees it maps to a real
action (consolidate, refresh, deprioritize). So "good" here means: silhouette holds up better
than a couple of naive splits (e.g. splitting only by `content_type` or `main_intent`), *and*
each cluster's median profile tells a story I can defend in one sentence.


In [6]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

numeric_feats = ["search_volume", "competition", "cpc", "word_count", "char_count",
                  "impressions_90d", "clicks_90d", "sessions_90d", "ctr",
                  "avg_position", "engagement_rate", "ai_traffic_pct"]
cat_feats = ["content_type", "main_intent"]

work = df[numeric_feats + cat_feats].copy()
work[numeric_feats] = work[numeric_feats].fillna(work[numeric_feats].median())
work[cat_feats] = work[cat_feats].fillna("unknown")

pre = ColumnTransformer([
    ("num", StandardScaler(), numeric_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])
X = pre.fit_transform(work)

# sample for a fast silhouette check while framing; full run happens in the capstone notebook
rng = np.random.default_rng(0)
sample_idx = rng.choice(X.shape[0], size=5000, replace=False)
X_sample = X[sample_idx]

for k in [3, 4, 5, 6]:
    labels = KMeans(n_clusters=k, random_state=0, n_init=10).fit_predict(X_sample)
    print(f"k={k}  silhouette={silhouette_score(X_sample, labels):.3f}")


k=3  silhouette=0.332
k=4  silhouette=0.334
k=5  silhouette=0.324
k=6  silhouette=0.334


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), which is the grain the whole starter dataset
ships in. I load the real 30k-row slice and confirm the shape below before touching anything.


In [7]:
df.head(3)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A fixed rule already exists here — `content_type` and `main_intent` — and it's too coarse.**
The data has exactly 4 intent buckets and 2-3 content-type buckets, an if-statement's worth
of categories. But the crosstab below shows my clusters cut *across* those buckets: each
cluster contains a mix of commercial, informational, and transactional pages. That means the
real groupings — the ones that actually separate high-performing, well-covered topics from
thin or cannibalized ones — live in the combination of many continuous, entangled signals
(search volume, word count, impressions, CTR, engagement) that don't line up with any single
rule anyone wrote by hand. That's exactly the case where clustering earns its place over a
lookup table.


In [8]:
# Fit k=4 and check whether clusters line up with the existing rule-based buckets
# (content_type / main_intent) or cut across them.
km = KMeans(n_clusters=4, random_state=0, n_init=10)
labels = km.fit_predict(X_sample)

sub = df.iloc[sample_idx].copy()
sub["cluster"] = labels

print(pd.crosstab(sub["cluster"], sub["main_intent"]))
print()
print(sub.groupby("cluster")[["search_volume", "word_count", "impressions_90d",
                               "ctr", "engagement_rate"]].median())


main_intent  commercial  informational  navigational  transactional
cluster                                                            
0                   109            433             2             87
1                   450           2275             2            592
2                     0              3             0              3
3                   195            169             4            323

         search_volume  word_count  impressions_90d   ctr  engagement_rate
cluster                                                                   
0                  0.0      5654.0           5730.0  0.14             0.75
1                 10.0      2787.0            620.0  0.06             0.00
2                 30.0      3108.0             25.5  0.00             0.00
3                 20.0      2677.0            492.0  0.03             0.00


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.